# Figure 4 t-SNE Sensitivity Analysis

This notebook isolates the Graphical Lasso t-SNE sensitivity checks from `make_main_figures_revision.ipynb`.

The baseline uses the original Euclidean profile-distance clustering. The later sections change one thing at a time, starting with the feature-profile distance metric.

## Setup

Load the same revision helpers and dataset cache used by the main figure notebook.

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]

pkg_root = repo_root / "data_synthesis"
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

import src.figure4_neighborhood as figure4_neighborhood
figure4_neighborhood = importlib.reload(figure4_neighborhood)

import src.main_figures_revision as figrev
figrev = importlib.reload(figrev)

EXPORT_DIR = repo_root / "data_synthesis" / "notebooks" / "revision_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {figrev.RUN_MODE} | CVAE_EPOCHS={figrev.CVAE_EPOCHS}")

In [ ]:
datasets = figrev.require_datasets()
figure4_real_data, figure4_synthetic_data, figure4_feature_names = figrev._get_figure4_precision_inputs()

pd.DataFrame({
    "dataset": list(figure4_real_data),
    "n_samples": [figure4_real_data[ds].shape[0] for ds in figure4_real_data],
    "n_features": [figure4_real_data[ds].shape[1] for ds in figure4_real_data],
    "alpha": [figrev.FIGURE4_ALPHAS[ds] for ds in figure4_real_data],
})

## Baseline: Euclidean Profile Distance

These are the original supplemental Graphical Lasso t-SNE edge-overlay layouts. The t-SNE coordinates are fit from real partial-correlation profiles, and feature cluster colors use average-linkage hierarchical clustering with Euclidean distance on those same profiles.

In [ ]:
baseline_tsne = {}
for dataset_name in figrev.DATASET_ORDER:
    result = figrev.plot_figure4_tsne_analysis_supplement(
        dataset_name,
        cluster_metric="euclidean",
        cluster_linkage="average",
    )
    baseline_tsne[dataset_name] = result
    figrev.display_result_once(result)

### Baseline Cluster Sizes

This summarizes the feature-cluster sizes induced by the baseline Euclidean profile-distance clustering. It does not use synthetic data; it is only about clustering real Graphical Lasso partial-correlation profiles.

In [ ]:
def real_profile_matrix(dataset_name):
    X = np.asarray(figure4_real_data[dataset_name], dtype=float)
    theta = figure4_neighborhood.fit_glasso_precision(X, figrev.FIGURE4_ALPHAS[dataset_name])
    partial = figure4_neighborhood.precision_to_partial_corr(theta)
    profiles = partial.copy()
    np.fill_diagonal(profiles, 0.0)
    profiles = StandardScaler().fit_transform(profiles)
    return partial, profiles


def heuristic_k(n_features, max_clusters=7):
    k = int(min(max_clusters, max(3, round(np.sqrt(n_features) * 1.35))))
    return min(k, max(2, n_features // 2))


def cluster_labels_for_profiles(profiles, metric="euclidean", linkage_method="average", max_clusters=7):
    n_features = profiles.shape[0]
    k = heuristic_k(n_features, max_clusters=max_clusters)
    distances = pdist(profiles, metric=metric)
    linkage = hierarchy.linkage(distances, method=linkage_method)
    return hierarchy.fcluster(linkage, t=k, criterion="maxclust")


def cluster_size_summary(metric="euclidean", linkage_method="average"):
    rows = []
    for dataset_name in figrev.DATASET_ORDER:
        _, profiles = real_profile_matrix(dataset_name)
        labels = cluster_labels_for_profiles(profiles, metric=metric, linkage_method=linkage_method)
        counts = pd.Series(labels).value_counts().sort_values(ascending=False).to_list()
        rows.append({
            "dataset": dataset_name,
            "metric": metric,
            "linkage": linkage_method,
            "n_features": profiles.shape[0],
            "requested_k": heuristic_k(profiles.shape[0]),
            "actual_k": len(set(labels)),
            "largest_cluster": counts[0],
            "cluster_sizes": ",".join(map(str, counts)),
        })
    return pd.DataFrame(rows)

baseline_cluster_sizes = cluster_size_summary(metric="euclidean", linkage_method="average")
display(baseline_cluster_sizes)

## Changing Distance

Everything in this section keeps the same real Graphical Lasso partial-correlation profiles and average-linkage clustering, but changes the distance metric between feature profiles.

This asks whether sparse dependency profiles should be clustered by absolute coordinate distance (`euclidean`) or by profile direction/pattern (`cosine` or `correlation`).

### Cosine Profile Distance

Cosine distance compares the direction of two sparse dependency-profile vectors. This can separate features that connect to similar biomarker neighborhoods even when their absolute edge strengths differ.

In [ ]:
cosine_tsne = {}
for dataset_name in figrev.DATASET_ORDER:
    result = figrev.plot_figure4_tsne_analysis_supplement(
        dataset_name,
        cluster_metric="cosine",
        cluster_linkage="average",
    )
    cosine_tsne[dataset_name] = result
    figrev.display_result_once(result)

### Compare Baseline vs Cosine Cluster Sizes

The same requested cluster count is used for both rows within each dataset. A large change in cluster sizes means the distance metric materially changes the grouping.

In [ ]:
distance_cluster_sizes = pd.concat([
    cluster_size_summary(metric="euclidean", linkage_method="average"),
    cluster_size_summary(metric="cosine", linkage_method="average"),
    cluster_size_summary(metric="correlation", linkage_method="average"),
], ignore_index=True)

display(distance_cluster_sizes.sort_values(["dataset", "metric"]))

### Full Distance/Linkage Sensitivity Table

This grid tries several linkage methods, distance metrics, and requested cluster counts. `silhouette` is a rough separation score; higher is better, but it should be interpreted alongside cluster sizes and biological plausibility.

In [ ]:
def cluster_sensitivity_table(max_requested_k=12):
    rows = []
    labels_by_dataset = {}
    settings = []
    for dataset_name in figrev.DATASET_ORDER:
        _, profiles = real_profile_matrix(dataset_name)
        n_features = profiles.shape[0]
        settings.clear()
        labels_by_setting = {}
        for linkage_method in ["single", "complete", "average", "ward"]:
            for metric in ["euclidean", "cosine", "correlation"]:
                if linkage_method == "ward" and metric != "euclidean":
                    continue
                distances = pdist(profiles, metric=metric)
                if not np.all(np.isfinite(distances)) or np.allclose(distances, 0):
                    continue
                linkage = hierarchy.linkage(distances, method=linkage_method)
                for requested_k in range(3, min(max_requested_k, max(3, n_features - 1)) + 1):
                    labels = hierarchy.fcluster(linkage, t=requested_k, criterion="maxclust")
                    actual_k = len(set(labels))
                    counts = pd.Series(labels).value_counts().sort_values(ascending=False).to_list()
                    silhouette = np.nan
                    if 1 < actual_k < n_features:
                        silhouette = silhouette_score(
                            profiles,
                            labels,
                            metric=metric if linkage_method != "ward" else "euclidean",
                        )
                    key = (linkage_method, metric, requested_k)
                    labels_by_setting[key] = labels
                    rows.append({
                        "dataset": dataset_name,
                        "n_features": n_features,
                        "linkage": linkage_method,
                        "metric": metric,
                        "requested_k": requested_k,
                        "actual_k": actual_k,
                        "largest_cluster": counts[0],
                        "cluster_sizes": ",".join(map(str, counts)),
                        "silhouette": silhouette,
                    })
        baseline_key = ("average", "euclidean", heuristic_k(n_features))
        baseline_labels = labels_by_setting.get(baseline_key)
        if baseline_labels is not None:
            for row in rows:
                if row["dataset"] != dataset_name or row["requested_k"] != baseline_key[2]:
                    continue
                key = (row["linkage"], row["metric"], row["requested_k"])
                row["ari_to_baseline"] = adjusted_rand_score(baseline_labels, labels_by_setting[key])
    return pd.DataFrame(rows)

sensitivity = cluster_sensitivity_table()
sensitivity_path = EXPORT_DIR / "figure4_tsne_distance_sensitivity.csv"
sensitivity.to_csv(sensitivity_path, index=False)
print(f"Wrote {sensitivity_path}")

display(
    sensitivity
    .sort_values(["dataset", "silhouette"], ascending=[True, False])
    .groupby("dataset")
    .head(10)
)

### Same-k Average-Linkage Distance Comparison

This table is the simplest apples-to-apples comparison: same requested `k`, same average linkage, only the distance metric changes.

In [ ]:
same_k_average = []
for dataset_name, sub in sensitivity.groupby("dataset", sort=False):
    k = heuristic_k(int(sub["n_features"].iloc[0]))
    same_k_average.append(
        sub[
            (sub["requested_k"] == k)
            & (sub["linkage"] == "average")
            & (sub["metric"].isin(["euclidean", "cosine", "correlation"]))
        ]
    )
same_k_average = pd.concat(same_k_average, ignore_index=True)
display(same_k_average[[
    "dataset",
    "metric",
    "requested_k",
    "actual_k",
    "largest_cluster",
    "cluster_sizes",
    "silhouette",
    "ari_to_baseline",
]])

## Changing t-SNE Perplexity

Everything in this section keeps the same high-dimensional feature clusters, but changes the t-SNE perplexity used to draw the 2D layout. This is useful for diagnosing layout artifacts, such as one profile cluster appearing as two separated orange islands in a single t-SNE panel. If a split moves around or disappears as perplexity changes, it is probably a projection artifact rather than a new cluster.

In [ ]:
PERPLEXITY_DATASET = "HIV"
PERPLEXITY_DISTANCE = "cosine"
PERPLEXITY_LINKAGE = "average"
PERPLEXITIES = [10, 20, 30, 40, 50]

partial, profiles = real_profile_matrix(PERPLEXITY_DATASET)
cluster_labels = figure4_neighborhood._profile_clusters(
    profiles,
    max_clusters=7,
    metric=PERPLEXITY_DISTANCE,
    linkage_method=PERPLEXITY_LINKAGE,
)
feature_names_for_plot = list(figure4_feature_names[PERPLEXITY_DATASET])

rows = []
for perplexity in PERPLEXITIES:
    coords, _, used_perplexity = figure4_neighborhood._fit_profile_tsne(
        partial,
        seed=figrev.SEED,
        perplexity=perplexity,
    )
    for feature_index, (x, y) in enumerate(coords):
        rows.append({
            "perplexity": int(used_perplexity),
            "feature_index": feature_index,
            "feature": feature_names_for_plot[feature_index],
            "tsne_1": x,
            "tsne_2": y,
            "profile_cluster": f"cluster {int(cluster_labels[feature_index])}",
            "cluster_id": int(cluster_labels[feature_index]),
        })
perplexity_df = pd.DataFrame(rows)

try:
    import plotly.express as px
    fig_perplexity = px.scatter(
        perplexity_df,
        x="tsne_1",
        y="tsne_2",
        color="profile_cluster",
        facet_col="perplexity",
        facet_col_wrap=3,
        hover_name="feature",
        hover_data={
            "feature_index": True,
            "profile_cluster": True,
            "cluster_id": False,
            "tsne_1": False,
            "tsne_2": False,
        },
        title=(
            f"{PERPLEXITY_DATASET}: t-SNE perplexity sensitivity | "
            f"clusters={PERPLEXITY_LINKAGE}+{PERPLEXITY_DISTANCE}"
        ),
        width=1150,
        height=820,
    )
    fig_perplexity.update_traces(marker=dict(size=8.5, opacity=0.86, line=dict(width=0.6, color="#222222")))
    fig_perplexity.update_xaxes(showgrid=False, zeroline=False, matches=None)
    fig_perplexity.update_yaxes(showgrid=False, zeroline=False, matches=None)
    fig_perplexity.show()
except Exception as exc:
    print(f"Plotly unavailable: {exc!r}")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.4))
    axes = axes.ravel()
    for ax, perplexity in zip(axes, PERPLEXITIES):
        sub = perplexity_df[perplexity_df["perplexity"] == perplexity]
        for cluster_name, group in sub.groupby("profile_cluster"):
            ax.scatter(group["tsne_1"], group["tsne_2"], s=34, label=cluster_name, alpha=0.86, edgecolor="#222222", linewidth=0.4)
        ax.set_title(f"perplexity={perplexity}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[-1].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=8)
    fig.tight_layout(rect=(0, 0, 0.88, 1))
    plt.show()

display(perplexity_df.head())

### Preserved / Lost / Synthetic-Only Summary with Prominent Features and Support Counts

This is a single cluster-summary plot for the distance setting below. Each cluster label combines the prominent feature(s) with the support count for the synthetic method assigned to that cluster, shown as `method: matching features / cluster features`. Cluster boundaries are hidden here because dot color already encodes profile-cluster membership; avoiding hulls keeps nearby or interleaved t-SNE groups from looking like fuzzy regions.


In [ ]:
SUMMARY_DISTANCE = "cosine"  # Options: "euclidean", "cosine", "correlation"
SUMMARY_LINKAGE = "average"
PROMINENT_FEATURES_PER_CLUSTER = 1

feature_support_summary = figure4_neighborhood.plot_figure4_cluster_summary_grid(
    real_data=figure4_real_data,
    synthetic_data=figure4_synthetic_data,
    feature_names=figure4_feature_names,
    alphas=figrev.FIGURE4_ALPHAS,
    dataset_order=["HIV", "Diabetes", "Breast Cancer"],
    method_order=figrev.METHOD_ORDER,
    cluster_metric=SUMMARY_DISTANCE,
    cluster_linkage=SUMMARY_LINKAGE,
    cluster_feature_label_top=PROMINENT_FEATURES_PER_CLUSTER,
    show_feature_support=True,
    cluster_blob_pad=0.0,
    cluster_boundary_style="none",
    cluster_fill_alpha=0.0,
    seed=figrev.SEED,
)
feature_support_group_summary = feature_support_summary.group_summary

figrev.display_result_once(feature_support_summary)
display(
    feature_support_group_summary[
        [
            "dataset",
            "panel",
            "mode",
            "cluster_id",
            "method",
            "n_features_matching_method",
            "n_features",
            "prominent_features",
        ]
    ].sort_values(["dataset", "panel", "cluster_id"])
)

### 2D Membership Diagnostic for Overlapping Groups

Use this when the publication-style Preserved/Lost/Synthetic-only panel makes it look like different colored dots are inside the same group. This diagnostic keeps the same 2D t-SNE coordinates, but makes membership explicit:

- point color = profile cluster ID
- point symbol = synthetic-method assignment for the selected mode
- hover = feature name, cluster ID, method assignment, and real Graphical Lasso degree

If two colors appear inside the same publication outline, check this plot: they are usually separate profile clusters whose method-colored envelopes overlap or share the same synthetic-method summary color.

In [ ]:
DIAG_DATASET = "HIV"  # Options: "HIV", "Breast Cancer", "Diabetes"
DIAG_MODE = "preserve"  # Options: "preserve", "lost", "synthetic_only"
DIAG_DISTANCE = "cosine"
DIAG_LINKAGE = "average"

assert DIAG_DATASET in figrev.DATASET_ORDER, figrev.DATASET_ORDER
assert DIAG_MODE in {"preserve", "lost", "synthetic_only"}

partial, profiles = real_profile_matrix(DIAG_DATASET)
coords2, _, perplexity = figure4_neighborhood._fit_profile_tsne(partial, seed=figrev.SEED)
cluster_labels = figure4_neighborhood._profile_clusters(
    profiles,
    max_clusters=7,
    metric=DIAG_DISTANCE,
    linkage_method=DIAG_LINKAGE,
)
clusters = figure4_neighborhood._clusters_from_labels(cluster_labels)
real_edges = figure4_neighborhood.get_edge_set(partial)

# Reuse the same synthetic edge sets used by the Figure 4 helper.
structures, _ = figure4_neighborhood._fit_structures(
    real_data=figure4_real_data,
    synthetic_data=figure4_synthetic_data,
    alphas=figrev.FIGURE4_ALPHAS,
    threshold=1e-7,
    dataset_order=[DIAG_DATASET],
    method_order=figrev.METHOD_ORDER,
)
synthetic_edge_map = {
    method: structures[DIAG_DATASET]["synthetic"][method]["edges"]
    for method in figrev.METHOD_ORDER
}
feature_scores = figure4_neighborhood.build_feature_preservation_scores(
    real_edges,
    synthetic_edge_map,
    profiles.shape[0],
)

if DIAG_MODE == "preserve":
    assignments, _ = figure4_neighborhood.summarize_feature_preservation(feature_scores, method_order=figrev.METHOD_ORDER)
    method_col = "best_method"
elif DIAG_MODE == "lost":
    assignments, _ = figure4_neighborhood.summarize_feature_loss(feature_scores, method_order=figrev.METHOD_ORDER)
    method_col = "lost_method"
else:
    assignments, _ = figure4_neighborhood.summarize_feature_synthetic_only(feature_scores, method_order=figrev.METHOD_ORDER)
    method_col = "synthetic_only_method"
method_by_feature = assignments.set_index("feature_index")[method_col]

cluster_summary = figure4_neighborhood._cluster_method_summary(
    feature_scores,
    clusters,
    figrev.METHOD_ORDER,
    mode=DIAG_MODE,
    feature_names=figure4_feature_names[DIAG_DATASET],
    top_features=1,
)
cluster_summary_lookup = cluster_summary.set_index("cluster_id")

names = list(figure4_feature_names[DIAG_DATASET])
degrees = feature_scores.drop_duplicates("feature_index").sort_values("feature_index")["real_degree"].to_numpy()
diag_df = pd.DataFrame({
    "feature_index": np.arange(profiles.shape[0]),
    "feature": names,
    "tsne_1": coords2[:, 0],
    "tsne_2": coords2[:, 1],
    "profile_cluster": [f"cluster {int(label)}" for label in cluster_labels],
    "cluster_id": cluster_labels.astype(int),
    "method_assignment": [method_by_feature.loc[i] for i in range(profiles.shape[0])],
    "real_degree": degrees,
})
diag_df["cluster_summary_method"] = diag_df["cluster_id"].map(cluster_summary_lookup["method"])
diag_df["cluster_support"] = diag_df["cluster_id"].map(
    lambda cid: f"{int(cluster_summary_lookup.loc[cid, 'n_features_matching_method'])}/{int(cluster_summary_lookup.loc[cid, 'n_features'])}"
)
diag_df["prominent_feature_for_cluster"] = diag_df["cluster_id"].map(cluster_summary_lookup["prominent_features"])
diag_df["point_size"] = 8 + 3.0 * np.sqrt(diag_df["real_degree"] + 1)

try:
    import plotly.express as px
    fig_diag = px.scatter(
        diag_df,
        x="tsne_1",
        y="tsne_2",
        color="profile_cluster",
        symbol="method_assignment",
        size="point_size",
        hover_name="feature",
        hover_data={
            "feature_index": True,
            "profile_cluster": True,
            "method_assignment": True,
            "cluster_summary_method": True,
            "cluster_support": True,
            "prominent_feature_for_cluster": True,
            "real_degree": True,
            "point_size": False,
            "tsne_1": False,
            "tsne_2": False,
            "cluster_id": False,
        },
        title=(
            f"{DIAG_DATASET}: 2D membership diagnostic | {DIAG_MODE}, "
            f"{DIAG_LINKAGE}+{DIAG_DISTANCE}, t-SNE perplexity={perplexity:.0f}"
        ),
        width=980,
        height=760,
    )
    fig_diag.update_traces(marker=dict(opacity=0.86, line=dict(width=0.8, color="#222222")))
    fig_diag.update_xaxes(showgrid=False, zeroline=False)
    fig_diag.update_yaxes(showgrid=False, zeroline=False, scaleanchor="x", scaleratio=1)
    fig_diag.show()
except Exception as exc:
    print(f"Plotly diagnostic unavailable: {exc!r}")
    display(diag_df.sort_values(["cluster_id", "real_degree"], ascending=[True, False]))

display(
    diag_df[
        [
            "feature",
            "cluster_id",
            "profile_cluster",
            "method_assignment",
            "cluster_summary_method",
            "cluster_support",
            "prominent_feature_for_cluster",
            "real_degree",
        ]
    ].sort_values(["cluster_id", "real_degree"], ascending=[True, False])
)

### 3D t-SNE Inspection for One Preserved/Lost/Synthetic Panel

The 2D summary plot uses two color vocabularies: dot color indicates the feature-profile cluster, while outline/text color indicates the synthetic-method summary for that cluster. This 3D view separates those ideas more explicitly. Use `COLOR_BY = "cluster"` to inspect profile clusters, or `COLOR_BY = "method_assignment"` to inspect which synthetic method each feature is assigned to for the selected mode.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

VIEW_DATASET = "HIV"  # Options: "HIV", "Breast Cancer", "Diabetes"
VIEW_MODE = "preserve"  # Options: "preserve", "lost", "synthetic_only"
VIEW_DISTANCE = "cosine"
VIEW_LINKAGE = "average"
COLOR_BY = "cluster"  # Options: "cluster", "method_assignment"

assert VIEW_DATASET in figrev.DATASET_ORDER, figrev.DATASET_ORDER
assert VIEW_MODE in {"preserve", "lost", "synthetic_only"}
assert COLOR_BY in {"cluster", "method_assignment"}

partial, profiles = real_profile_matrix(VIEW_DATASET)
coords3_kwargs = dict(
    n_components=3,
    perplexity=min(30, max(2, (profiles.shape[0] - 1) // 3)),
    init="pca",
    learning_rate="auto",
    random_state=figrev.SEED,
    metric="euclidean",
)
try:
    coords3 = TSNE(max_iter=1500, **coords3_kwargs).fit_transform(profiles)
except TypeError:
    coords3 = TSNE(n_iter=1500, **coords3_kwargs).fit_transform(profiles)

cluster_labels = figure4_neighborhood._profile_clusters(
    profiles,
    max_clusters=7,
    metric=VIEW_DISTANCE,
    linkage_method=VIEW_LINKAGE,
)
clusters = figure4_neighborhood._clusters_from_labels(cluster_labels)
real_edges = figure4_neighborhood.get_edge_set(partial)
synthetic_edge_map = {
    method: selected_tsne_grouping.structures[VIEW_DATASET]["synthetic"][method]["edges"]
    if "selected_tsne_grouping" in globals() and VIEW_DATASET in selected_tsne_grouping.structures
    else figure4_neighborhood.get_edge_set(
        figure4_neighborhood.precision_to_partial_corr(
            figure4_neighborhood.fit_glasso_precision(
                figure4_synthetic_data[VIEW_DATASET][method],
                figrev.FIGURE4_ALPHAS[VIEW_DATASET],
            )
        )
    )
    for method in figrev.METHOD_ORDER
}
feature_scores = figure4_neighborhood.build_feature_preservation_scores(
    real_edges,
    synthetic_edge_map,
    profiles.shape[0],
)

if VIEW_MODE == "preserve":
    assignments, _ = figure4_neighborhood.summarize_feature_preservation(feature_scores, method_order=figrev.METHOD_ORDER)
    method_by_feature = assignments.set_index("feature_index")["best_method"]
elif VIEW_MODE == "lost":
    assignments, _ = figure4_neighborhood.summarize_feature_loss(feature_scores, method_order=figrev.METHOD_ORDER)
    method_by_feature = assignments.set_index("feature_index")["lost_method"]
else:
    assignments, _ = figure4_neighborhood.summarize_feature_synthetic_only(feature_scores, method_order=figrev.METHOD_ORDER)
    method_by_feature = assignments.set_index("feature_index")["synthetic_only_method"]

names = list(figure4_feature_names[VIEW_DATASET])
degrees = feature_scores.drop_duplicates("feature_index").sort_values("feature_index")["real_degree"].to_numpy()
plot_df = pd.DataFrame({
    "feature_index": np.arange(profiles.shape[0]),
    "feature": names,
    "x": coords3[:, 0],
    "y": coords3[:, 1],
    "z": coords3[:, 2],
    "cluster": [f"cluster {int(label)}" for label in cluster_labels],
    "method_assignment": [method_by_feature.loc[i] for i in range(profiles.shape[0])],
    "real_degree": degrees,
})
plot_df["point_size"] = 28 + 16 * np.sqrt(plot_df["real_degree"] + 1)

title = (
    f"{VIEW_DATASET}: 3D t-SNE of real Graphical Lasso profiles | "
    f"{VIEW_MODE}, {VIEW_LINKAGE}+{VIEW_DISTANCE}, color={COLOR_BY}"
)

try:
    import plotly.express as px
    fig3d = px.scatter_3d(
        plot_df,
        x="x",
        y="y",
        z="z",
        color=COLOR_BY,
        size="point_size",
        hover_name="feature",
        hover_data={
            "feature_index": True,
            "cluster": True,
            "method_assignment": True,
            "real_degree": True,
            "point_size": False,
            "x": False,
            "y": False,
            "z": False,
        },
        title=title,
        width=920,
        height=760,
    )
    fig3d.update_traces(marker=dict(opacity=0.86, line=dict(width=0.7, color="#222222")))
    fig3d.show()
except Exception as exc:
    print("Plotly is unavailable; using Matplotlib mplot3d fallback.")
    print(f"Install plotly for hover/rotation in-browser: {exc!r}")
    categories = pd.Categorical(plot_df[COLOR_BY])
    cmap = plt.get_cmap("tab10", len(categories.categories))
    fig = plt.figure(figsize=(8.8, 7.4))
    ax = fig.add_subplot(111, projection="3d")
    for code_value, category in enumerate(categories.categories):
        sub = plot_df[categories == category]
        ax.scatter(
            sub["x"],
            sub["y"],
            sub["z"],
            s=sub["point_size"],
            color=cmap(code_value),
            edgecolor="#222222",
            linewidth=0.45,
            alpha=0.86,
            label=str(category),
        )
    top_labels = plot_df.sort_values("real_degree", ascending=False).head(12)
    for _, row in top_labels.iterrows():
        ax.text(row["x"], row["y"], row["z"], figure4_neighborhood._short_label(row["feature"], 14), fontsize=7)
    ax.set_title(title, pad=14)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_zlabel("t-SNE 3")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8)
    plt.tight_layout()
    plt.show()

display(plot_df.sort_values([COLOR_BY, "real_degree"], ascending=[True, False]).head(30))

## Euclidean vs Cosine Across t-SNE Perplexity

This plot compares the visual layout produced across perplexities 10-50 for both Euclidean-distance and cosine-distance profile clustering. The t-SNE coordinates are recomputed for each perplexity from the same signed Graphical Lasso profiles; colors show cluster labels from the selected distance metric.

In [ ]:
COMPARE_DATASET = "HIV"  # Options: "HIV", "Breast Cancer", "Diabetes"
COMPARE_LINKAGE = "average"
COMPARE_PERPLEXITIES = [10, 20, 30, 40, 50]
COMPARE_METRICS = ["euclidean", "cosine"]

partial, profiles = real_profile_matrix(COMPARE_DATASET)
feature_names_compare = list(figure4_feature_names[COMPARE_DATASET])
compare_rows = []
for metric in COMPARE_METRICS:
    cluster_labels = figure4_neighborhood._profile_clusters(
        profiles,
        max_clusters=7,
        metric=metric,
        linkage_method=COMPARE_LINKAGE,
    )
    for perplexity in COMPARE_PERPLEXITIES:
        coords, _, used_perplexity = figure4_neighborhood._fit_profile_tsne(
            partial,
            seed=figrev.SEED,
            perplexity=perplexity,
        )
        for feature_index, (x, y) in enumerate(coords):
            compare_rows.append({
                "metric": metric,
                "perplexity": int(used_perplexity),
                "feature_index": feature_index,
                "feature": feature_names_compare[feature_index],
                "tsne_1": x,
                "tsne_2": y,
                "cluster_id": int(cluster_labels[feature_index]),
                "profile_cluster": f"cluster {int(cluster_labels[feature_index])}",
            })
compare_perplexity_df = pd.DataFrame(compare_rows)

import matplotlib.pyplot as plt
cluster_ids = sorted(compare_perplexity_df["cluster_id"].unique())
colors = dict(zip(cluster_ids, plt.get_cmap("tab10")(np.linspace(0, 1, max(len(cluster_ids), 1)))))
fig, axes = plt.subplots(
    len(COMPARE_METRICS),
    len(COMPARE_PERPLEXITIES),
    figsize=(3.15 * len(COMPARE_PERPLEXITIES), 3.0 * len(COMPARE_METRICS)),
    constrained_layout=False,
)
axes = np.asarray(axes)
if axes.ndim == 1:
    axes = axes.reshape(len(COMPARE_METRICS), len(COMPARE_PERPLEXITIES))

for row_idx, metric in enumerate(COMPARE_METRICS):
    for col_idx, perplexity in enumerate(COMPARE_PERPLEXITIES):
        ax = axes[row_idx, col_idx]
        sub = compare_perplexity_df[
            (compare_perplexity_df["metric"] == metric)
            & (compare_perplexity_df["perplexity"] == perplexity)
        ]
        for cluster_id, group in sub.groupby("cluster_id", sort=True):
            ax.scatter(
                group["tsne_1"],
                group["tsne_2"],
                s=28,
                color=colors[cluster_id],
                edgecolor="#222222",
                linewidth=0.35,
                alpha=0.86,
            )
        if row_idx == 0:
            ax.set_title(f"perplexity={perplexity}", fontsize=10.5, weight="semibold")
        if col_idx == 0:
            ax.set_ylabel(metric.title(), fontsize=10.5, weight="semibold")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.75)
            spine.set_edgecolor("#333333")

handles = [
    plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=colors[cid], markeredgecolor="#222222", markersize=6, label=f"cluster {cid}")
    for cid in cluster_ids
]
fig.legend(handles=handles, loc="center right", bbox_to_anchor=(1.015, 0.5), fontsize=8, title="Profile cluster")
fig.suptitle(
    f"{COMPARE_DATASET}: Euclidean vs cosine clustering across t-SNE perplexity ({COMPARE_LINKAGE} linkage)",
    fontsize=13.0,
    weight="semibold",
    y=0.985,
)
fig.subplots_adjust(left=0.045, right=0.90, top=0.90, bottom=0.055, wspace=0.08, hspace=0.12)
plt.show()

display(
    compare_perplexity_df
    .groupby(["metric", "cluster_id"], as_index=False)
    .agg(n_features=("feature_index", "nunique"))
    .sort_values(["metric", "n_features"], ascending=[True, False])
)

## Working Interpretation

Use this section to decide whether the supplemental figure should retain Euclidean cluster colors or switch/add cosine cluster-color sensitivity versions.

Current working read:

- HIV: cosine distance materially changes the clustering and avoids the large Euclidean 55-feature cluster.
- Breast Cancer: cosine/correlation also improve separation and produce more balanced clusters.
- Diabetes: distance changes the silhouette score but not the same-k cluster sizes, so the visual grouping is less sensitive.